# Analyse des indicateurs environnementaux

Ce notebook prépare `data/raw/indicator_Data.csv`, un export des World Development Indicators.

Chaque ligne décrit **un indicateur pour un pays**. Les années 1960 à 2025 occupent les colonnes, et les valeurs manquantes sont codées `..`. Les dernières lignes sont un pied de page de la Banque mondiale.

On en tire deux tableaux :

- `long` : une observation par pays, indicateur et année
- `tidy` : une ligne par pays et année, avec une colonne par indicateur

Dépendances : `pandas` et `matplotlib`.

```bash
pip install pandas matplotlib
```

In [ ]:
from pathlib import Path

import pandas as pd


def find_indicator_csv() -> Path:
    relative = Path("data") / "raw" / "indicator_Data.csv"
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidate = base / relative
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        "indicator_Data.csv introuvable. Lancer le notebook depuis le dépôt planet-pulse."
    )


csv_path = find_indicator_csv()
raw = pd.read_csv(csv_path, na_values="..")

csv_path, raw.shape

## Format d'origine

Les quatre premières colonnes identifient le pays et la série. Tout le reste est une année. Le pied de page (`Data from database`, `Last Updated`) n'a pas de code pays.

In [ ]:
raw.head(3)

In [ ]:
raw.tail(5)

## Nettoyage

On retire le pied de page, puis on liste les séries réellement présentes dans le fichier.

In [ ]:
raw = raw.dropna(subset=["Country Code", "Series Code"]).copy()

series = (
    raw[["Series Code", "Series Name"]]
    .drop_duplicates()
    .sort_values("Series Code")
    .reset_index(drop=True)
)

series

## Format long

`melt` rassemble les colonnes d'années dans deux colonnes, `year` et `value`. L'en-tête `1990 [YR1990]` devient l'entier `1990`.

In [ ]:
id_cols = ["Country Name", "Country Code", "Series Name", "Series Code"]
year_cols = [column for column in raw.columns if column not in id_cols]

long = raw.melt(
    id_vars=id_cols,
    value_vars=year_cols,
    var_name="year",
    value_name="value",
)

long["year"] = long["year"].str.extract(r"(\d{4})").astype(int)
long["value"] = pd.to_numeric(long["value"], errors="coerce")
long = long.rename(
    columns={
        "Country Name": "country_name",
        "Country Code": "country_code",
        "Series Name": "series_name",
        "Series Code": "series_code",
    }
)

long.head()

In [ ]:
pd.Series(
    {
        "observations": len(long),
        "entites": long["country_code"].nunique(),
        "indicateurs": long["series_code"].nunique(),
        "annees": f"{long['year'].min()}–{long['year'].max()}",
        "valeurs_manquantes": f"{long['value'].isna().mean():.0%}",
    },
    name="format long",
)

## Une colonne par indicateur

Chaque ligne de `tidy` est un pays pour une année. Les codes Banque mondiale deviennent des noms de colonnes courts.

In [ ]:
SERIES_COLUMNS = {
    "EN.ATM.PM25.MC.M3": "pm25_ug_m3",
    "EN.GHG.CO2.ZG.AR5": "co2_change_vs_1990_pct",
    "AG.LND.FRST.ZS": "forest_area_pct",
    "EG.FEC.RNEW.ZS": "renewable_energy_pct",
}

tidy = (
    long.pivot(
        index=["country_name", "country_code", "year"],
        columns="series_code",
        values="value",
    )
    .reset_index()
    .rename(columns=SERIES_COLUMNS)
)
tidy.columns.name = None
tidy = tidy.sort_values(["country_name", "year"]).reset_index(drop=True)

tidy.head()

## Exemples

Le fichier mélange des pays et des agrégats Banque mondiale (`WLD` pour le monde, groupes de revenu, régions). Pour un pays, filtrer `country_code`.

In [ ]:
tidy.loc[tidy["country_code"].eq("FRA") & tidy["year"].eq(2020)]

In [ ]:
france = tidy.loc[tidy["country_code"].eq("FRA")].set_index("year")

ax = france["renewable_energy_pct"].dropna().plot(
    title="Part d'énergie renouvelable — France",
    ylabel="% de la consommation finale",
    xlabel="Année",
)
ax.figure.tight_layout()

In [ ]:
sample = ["FRA", "DEU", "BRA", "IND", "USA"]

forest = (
    tidy.loc[tidy["country_code"].isin(sample)]
    .pivot(index="year", columns="country_name", values="forest_area_pct")
    .dropna(how="all")
)

ax = forest.plot(
    title="Surface forestière",
    ylabel="% du territoire",
    xlabel="Année",
)
ax.figure.tight_layout()